# combine schroedinger pdbs of ligands with protein pdb

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.9 MB/s eta 0:00:00


In [1]:
import os
import sys
import glob
import re
import tempfile
from pathlib import Path
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from collections import defaultdict




In [5]:
def read_pdb(file_path):
    """Read a PDB file and return its content as a list of lines."""
    with open(file_path, 'r') as f:
        return [line.rstrip() for line in f if not line.startswith('END')]

def write_pdb(file_path, content):
    """Write PDB content to a file, adding END at the end."""
    with open(file_path, 'w') as f:
        for line in content:
            f.write(line + '\n')
        f.write('END\n')  # Add END record at the end of the file

def combine_pdbs(protein_file, ligand_files, output_pattern):
    """Combine a protein PDB with multiple ligand PDBs."""
    print(f"Reading protein from: {protein_file}")
    protein_content = read_pdb(protein_file)
    
    for i, ligand_file in enumerate(ligand_files, 1):
        print(f"Processing ligand {i}: {ligand_file}")
        ligand_content = read_pdb(ligand_file)
        
        # Determine output filename by replacing * with an index number
        if '*' in output_pattern:
            output_file = output_pattern.replace('*', str(i))
        else:
            ligand_stem = Path(ligand_file).stem  # Get filename without extension
            output_file = output_pattern.replace('*', ligand_stem)
        
        # Combine protein and ligand content
        combined_content = protein_content + ligand_content
        
        write_pdb(output_file, combined_content)
        print(f"  --> Created combined file: {output_file}")



def combine_with_biopython(protein_file, ligand_files, output_pattern):
    # Set up parser and IO
    parser = PDBParser(QUIET=True)
    io = PDBIO()
    
    # Load the protein structure
    protein_structure = parser.get_structure("protein", protein_file)
    
    for i, ligand_file in enumerate(ligand_files, 1):
        # Load the ligand structure
        ligand_structure = parser.get_structure("ligand", ligand_file)
        
        # Make a copy of the protein structure for this combination
        combined = protein_structure.copy()
        
        # Get the first model from each structure
        protein_model = combined[0]
        ligand_model = ligand_structure[0]
        
        # Create a new chain ID for the ligand (e.g., 'L')
        ligand_chain_id = 'L'
        
        # Add all ligand atoms to the protein model with the new chain ID
        for ligand_chain in ligand_model:
            for residue in ligand_chain:
                # Create a new residue in the protein model
                new_residue = residue.copy()
                # Add the new residue to the protein model with ligand chain ID
                if ligand_chain_id not in protein_model:
                    protein_model.add(PDB.Chain.Chain(ligand_chain_id))
                protein_model[ligand_chain_id].add(new_residue)
        
        # Determine output filename
        if '*' in output_pattern:
            output_file = output_pattern.replace('*', str(i))
        else:
            output_file = output_pattern.replace('*', f"complex_{i}")
        
        # Save the combined structure
        io.set_structure(combined)
        io.save(output_file)
        print(f"Created combined file: {output_file}")


In [6]:
class ConectRecord:
    """Class to store and manipulate CONECT records."""
    def __init__(self, source_atom, target_atoms):
        self.source_atom = source_atom
        self.target_atoms = target_atoms
    
    def __str__(self):
        """Format as a valid PDB CONECT record."""
        record = f"CONECT{self.source_atom:5d}"
        for atom in self.target_atoms:
            record += f"{atom:5d}"
        return record

def extract_conect_records(pdb_file):
    """Extract CONECT records from a PDB file and return as a list of ConectRecord objects."""
    conect_records = []
    conect_pattern = re.compile(r'CONECT\s+(\d+)(.+)')
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("CONECT"):
                match = conect_pattern.match(line)
                if match:
                    source_atom = int(match.group(1))
                    # Extract all target atoms
                    target_atoms = []
                    targets_str = match.group(2)
                    # Extract 5-character chunks for each atom
                    for i in range(0, len(targets_str), 5):
                        chunk = targets_str[i:i+5].strip()
                        if chunk:
                            try:
                                target_atoms.append(int(chunk))
                            except ValueError:
                                pass
                    
                    conect_records.append(ConectRecord(source_atom, target_atoms))
    
    return conect_records

def get_atom_serial_mapping(old_structure, new_structure):
    """
    Create a mapping from old atom serial numbers to new atom serial numbers.
    
    This is more comprehensive than the previous approach and works by matching
    atom coordinates which are preserved during structure copying.
    
    Args:
        old_structure: Original structure with original atom serials
        new_structure: New structure where atoms have been added with potentially new serials
        
    Returns:
        Dictionary mapping old serial numbers to new serial numbers
    """
    serial_mapping = {}
    
    # Create a dictionary of (x,y,z) coordinates to atom serial in old structure
    old_coords_to_serial = {}
    for model in old_structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # Get coordinates with some precision limit to avoid floating point issues
                    coords = tuple(round(coord, 3) for coord in atom.get_coord())
                    old_coords_to_serial[coords] = atom.serial_number
    
    # Match atoms in the new structure by their coordinates
    for model in new_structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    coords = tuple(round(coord, 3) for coord in atom.get_coord())
                    if coords in old_coords_to_serial:
                        old_serial = old_coords_to_serial[coords]
                        serial_mapping[old_serial] = atom.serial_number
    
    return serial_mapping

class SelectProtein(Select):
    """Select protein atoms for writing to PDB."""
    def accept_atom(self, atom):
        return atom.parent.id[0] != 'H_NEW'  # Skip new hetero atoms (will be added separately)

class SelectLigand(Select):
    """Select ligand atoms for writing to PDB."""
    def __init__(self, chain_id):
        self.chain_id = chain_id
    
    def accept_atom(self, atom):
        return atom.parent.parent.id == self.chain_id

class ConectPreservingPDBIO(PDBIO):
    """Extension of PDBIO that preserves CONECT records."""
    def __init__(self):
        super().__init__()
        self.conect_records = []
    
    def set_conect_records(self, records):
        """Set the CONECT records to be written out."""
        self.conect_records = records
    
    def save(self, file, select=None):
        """Save structure to a PDB file and append CONECT records before the END record."""
        # If no select is provided, use one that accepts everything
        if select is None:
            class SelectAll(Select):
                def accept_atom(self, atom): return True
                def accept_residue(self, residue): return True
                def accept_chain(self, chain): return True
                def accept_model(self, model): return True
            select = SelectAll()
        
        # First, write the structure to a temporary file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
            temp_name = temp_file.name
            super().save(temp_name, select)
        
        # Now read the temporary file, insert CONECT records before END, and write to the final file
        with open(temp_name, 'r') as temp_file, open(file, 'w') as out_file:
            # Read all lines except END
            lines = []
            for line in temp_file:
                if not line.startswith('END'):
                    lines.append(line)
            
            # Write all content except END
            for line in lines:
                out_file.write(line)
            
            # Write CONECT records
            if self.conect_records:
                for record in self.conect_records:
                    out_file.write(str(record) + '\n')
            
            # Write END record
            out_file.write('END\n')
        
        # Remove the temporary file
        import os
        os.unlink(temp_name)

def combine_structures_with_conect(protein_file, ligand_files, output_pattern):
    """
    Combine protein and ligand structures using BioPython while preserving CONECT records.
    
    Args:
        protein_file (str): Path to protein PDB file
        ligand_files (list): List of paths to ligand PDB files
        output_pattern (str): Pattern for output filenames
    """
    parser = PDBParser(QUIET=True)
    io = ConectPreservingPDBIO()
    
    # Extract protein structure and CONECT records
    protein_structure = parser.get_structure("protein", protein_file)
    protein_conect = extract_conect_records(protein_file)
    
    # Process each ligand
    for i, ligand_file in enumerate(ligand_files, 1):
        print(f"Processing ligand {i}: {ligand_file}")
        
        # Parse ligand structure and extract CONECT records
        ligand_structure = parser.get_structure("ligand", ligand_file)
        ligand_conect = extract_conect_records(ligand_file)
        
        # Create a new structure for the complex
        complex_structure = Structure("complex")
        complex_model = Model(0)
        complex_structure.add(complex_model)
        
        # Add all protein chains to the complex
        for chain in protein_structure[0]:
            complex_model.add(chain.copy())
        
        # Find an available chain ID for the ligand
        used_chain_ids = [chain.id for chain in complex_model]
        ligand_chain_id = 'X'  # Default
        for possible_id in "LXYZABCDEFGHIJKMNOPQRSTUVW":
            if possible_id not in used_chain_ids:
                ligand_chain_id = possible_id
                break
        
        # Create a new chain for the ligand
        ligand_chain = Chain(ligand_chain_id)
        
        # Add all ligand residues to the new chain
        for residue in ligand_structure[0].get_residues():
            ligand_chain.add(residue.copy())
        
        # Add the ligand chain to the complex
        complex_model.add(ligand_chain)
        
        # Get atom serial mappings for both protein and ligand
        protein_serial_mapping = get_atom_serial_mapping(protein_structure, complex_structure)
        ligand_serial_mapping = get_atom_serial_mapping(ligand_structure, complex_structure)
        
        # Print some debugging information
        print(f"  Found {len(protein_serial_mapping)} mappings for protein atoms")
        print(f"  Found {len(ligand_serial_mapping)} mappings for ligand atoms")
        
        # Debug: show some example mappings if available
        if ligand_conect and ligand_serial_mapping:
            sample_old_serials = list(ligand_serial_mapping.keys())[:5]
            print(f"  Sample ligand atom mappings (old -> new):")
            for old_serial in sample_old_serials:
                print(f"    {old_serial} -> {ligand_serial_mapping.get(old_serial, 'Not found')}")
        
        # Adjust CONECT records with new serial numbers
        adjusted_protein_conect = []
        for record in protein_conect:
            if record.source_atom in protein_serial_mapping:
                new_source = protein_serial_mapping[record.source_atom]
                new_targets = []
                for target in record.target_atoms:
                    if target in protein_serial_mapping:
                        new_targets.append(protein_serial_mapping[target])
                if new_targets:  # Only add if there are valid targets
                    adjusted_protein_conect.append(ConectRecord(new_source, new_targets))
        
        adjusted_ligand_conect = []
        for record in ligand_conect:
            if record.source_atom in ligand_serial_mapping:
                new_source = ligand_serial_mapping[record.source_atom]
                new_targets = []
                for target in record.target_atoms:
                    if target in ligand_serial_mapping:
                        new_targets.append(ligand_serial_mapping[target])
                if new_targets:  # Only add if there are valid targets
                    adjusted_ligand_conect.append(ConectRecord(new_source, new_targets))
        
        print(f"  Adjusted {len(adjusted_protein_conect)} protein CONECT records")
        print(f"  Adjusted {len(adjusted_ligand_conect)} ligand CONECT records")
        
        # Combine CONECT records
        all_conect = adjusted_protein_conect + adjusted_ligand_conect
        
        # Determine output filename
        if '*' in output_pattern:
            output_file = output_pattern.replace('*', str(i))
        else:
            ligand_stem = Path(ligand_file).stem
            output_file = output_pattern.replace('*', ligand_stem)
        
        # Set structure and CONECT records in the IO object
        io.set_structure(complex_structure)
        io.set_conect_records(all_conect)
        
        # Save the combined structure with CONECT records
        io.save(output_file)
        print(f"  --> Created combined file: {output_file}")


In [9]:
import os
import sys
import glob
import re
from pathlib import Path
from collections import defaultdict

from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain

class ConectRecord:
    """Class to store and manipulate CONECT records."""
    def __init__(self, source_atom, target_atoms):
        self.source_atom = source_atom
        self.target_atoms = target_atoms
    
    def __str__(self):
        """Format as a valid PDB CONECT record."""
        record = f"CONECT{self.source_atom:5d}"
        for atom in self.target_atoms:
            record += f"{atom:5d}"
        return record

def extract_conect_records(pdb_file):
    """Extract CONECT records from a PDB file and return as a list of ConectRecord objects."""
    conect_records = []
    conect_pattern = re.compile(r'CONECT\s+(\d+)(.+)')
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("CONECT"):
                match = conect_pattern.match(line)
                if match:
                    source_atom = int(match.group(1))
                    # Extract all target atoms
                    target_atoms = []
                    targets_str = match.group(2)
                    # Extract 5-character chunks for each atom
                    for i in range(0, len(targets_str), 5):
                        chunk = targets_str[i:i+5].strip()
                        if chunk:
                            try:
                                target_atoms.append(int(chunk))
                            except ValueError:
                                pass
                    
                    conect_records.append(ConectRecord(source_atom, target_atoms))
    
    return conect_records

class SelectProtein(Select):
    """Select protein atoms for writing to PDB."""
    def accept_atom(self, atom):
        return atom.parent.id[0] != 'H_NEW'  # Skip new hetero atoms (will be added separately)

class SelectLigand(Select):
    """Select ligand atoms for writing to PDB."""
    def __init__(self, chain_id):
        self.chain_id = chain_id
    
    def accept_atom(self, atom):
        return atom.parent.parent.id == self.chain_id

class ConectPreservingPDBIO(PDBIO):
    """Extension of PDBIO that preserves CONECT records."""
    def __init__(self):
        super().__init__()
        self.conect_records = []
    
    def set_conect_records(self, records):
        """Set the CONECT records to be written out."""
        self.conect_records = records
    
    def save(self, file, select=None):
        """Save structure to a PDB file and append CONECT records before the END record."""
        # If no select is provided, use one that accepts everything
        if select is None:
            class SelectAll(Select):
                def accept_atom(self, atom): return True
                def accept_residue(self, residue): return True
                def accept_chain(self, chain): return True
                def accept_model(self, model): return True
            select = SelectAll()
        
        # First, write the structure to a temporary file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
            temp_name = temp_file.name
            super().save(temp_name, select)
        
        # Now read the temporary file, insert CONECT records before END, and write to the final file
        with open(temp_name, 'r') as temp_file, open(file, 'w') as out_file:
            # Read all lines except END
            lines = []
            for line in temp_file:
                if not line.startswith('END'):
                    lines.append(line)
            
            # Write all content except END
            for line in lines:
                out_file.write(line)
            
            # Write CONECT records
            if self.conect_records:
                for record in self.conect_records:
                    out_file.write(str(record) + '\n')
            
            # Write END record
            out_file.write('END\n')
        
        # Remove the temporary file
        import os
        os.unlink(temp_name)

def extract_atoms_from_pdb(pdb_file):
    """
    Extract atom information directly from PDB file.
    Returns a dictionary mapping serial numbers to (x,y,z) coordinates.
    """
    atoms = {}
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    serial = int(line[6:11].strip())
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                    atoms[serial] = (round(x, 3), round(y, 3), round(z, 3))
                except (ValueError, IndexError):
                    pass
    return atoms

def extract_atom_mapping_from_output(input_file, output_file):
    """
    Extract mapping between input and output atom serials.
    This compares the actual PDB files to create a reliable mapping.
    """
    # Extract coordinates from input file with original serial numbers
    input_atoms = extract_atoms_from_pdb(input_file)
    
    # Extract coordinates from output file with new serial numbers
    output_atoms = {}
    coord_to_serial = {}
    
    with open(output_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    serial = int(line[6:11].strip())
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                    coords = (round(x, 3), round(y, 3), round(z, 3))
                    output_atoms[serial] = coords
                    coord_to_serial[coords] = serial
                except (ValueError, IndexError):
                    pass
    
    # Create mapping from input serial to output serial based on matching coordinates
    serial_mapping = {}
    for old_serial, coords in input_atoms.items():
        if coords in coord_to_serial:
            serial_mapping[old_serial] = coord_to_serial[coords]
    
    return serial_mapping

def combine_pdbs_direct(protein_file, ligand_file, output_file):
    """
    Combine protein and ligand PDB files directly without BioPython
    and properly adjust CONECT records.
    """
    # First just combine the files without handling CONECT records
    with open(protein_file, 'r') as p_file, open(ligand_file, 'r') as l_file, open(output_file, 'w') as out_file:
        # Write header
        out_file.write("REMARK   Generated by PDB Combiner Script\n")
        out_file.write(f"REMARK   Combines {protein_file} with {ligand_file}\n")
        
        # Write protein content (excluding END and CONECT)
        protein_lines = []
        for line in p_file:
            if not (line.startswith('END') or line.startswith('CONECT')):
                protein_lines.append(line)
        
        for line in protein_lines:
            out_file.write(line)
        
        # Find highest atom serial in protein to offset ligand atoms
        max_serial = 0
        for line in protein_lines:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    serial = int(line[6:11].strip())
                    max_serial = max(max_serial, serial)
                except (ValueError, IndexError):
                    pass
        
        print(f"  Highest protein atom serial: {max_serial}")
        
        # Create mapping for ligand atom serials
        ligand_mapping = {}
        ligand_lines = []
        
        for line in l_file:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    old_serial = int(line[6:11].strip())
                    new_serial = old_serial + max_serial
                    ligand_mapping[old_serial] = new_serial
                    
                    # Renumber the atom and update the line
                    new_line = line[:6] + f"{new_serial:5d}" + line[11:]
                    ligand_lines.append(new_line)
                except (ValueError, IndexError):
                    ligand_lines.append(line)
            elif not (line.startswith('END') or line.startswith('CONECT')):
                ligand_lines.append(line)
        
        # Write ligand content
        for line in ligand_lines:
            out_file.write(line)
        
        # Add TER record
        out_file.write("TER\n")
        
        # Process CONECT records from protein
        protein_conect = extract_conect_records(protein_file)
        for record in protein_conect:
            out_file.write(str(record) + '\n')
        
        # Process CONECT records from ligand with adjusted serial numbers
        ligand_conect = extract_conect_records(ligand_file)
        for record in ligand_conect:
            if record.source_atom in ligand_mapping:
                new_source = ligand_mapping[record.source_atom]
                new_targets = []
                for target in record.target_atoms:
                    if target in ligand_mapping:
                        new_targets.append(ligand_mapping[target])
                
                if new_targets:
                    new_record = ConectRecord(new_source, new_targets)
                    out_file.write(str(new_record) + '\n')
        
        # Write END
        out_file.write("END\n")
    
    print(f"  --> Created combined file: {output_file}")
    return ligand_mapping

def combine_structures_direct(protein_file, ligand_files, output_pattern):
    """
    Combine protein PDB with multiple ligand PDBs using direct file manipulation
    instead of BioPython, which has proven more reliable for CONECT records.
    """
    for i, ligand_file in enumerate(ligand_files, 1):
        print(f"Processing ligand {i}: {ligand_file}")
        
        # Determine output filename
        if '*' in output_pattern:
            output_file = output_pattern.replace('*', str(i))
        else:
            ligand_stem = Path(ligand_file).stem
            output_file = output_pattern.replace('*', ligand_stem)
        
        # Combine files directly
        ligand_mapping = combine_pdbs_direct(protein_file, ligand_file, output_file)
        
        # Print some sample mappings for verification
        sample_keys = list(ligand_mapping.keys())[:5]
        print("  Sample atom mappings (old -> new):")
        for old_serial in sample_keys:
            print(f"    {old_serial} -> {ligand_mapping[old_serial]}")
        
        print(f"  Mapped {len(ligand_mapping)} ligand atoms")

In [16]:
def create_combined_res(prot_file, protein_path, ligand_pattern, ligand_path, output_pattern, output_path):
    #check if user gave empty vars
    if not all([prot_file, protein_path, ligand_pattern, ligand_path, output_pattern, output_path]):
        print("Error: One or more required variables are empty!")
        return 0
    else:
        ##### not changes needed after that mark
        # Join the path and filename
        full_path_prot = os.path.join(protein_path, prot_file)
        full_pattern_lig = os.path.join(ligand_path, ligand_pattern)
        full_pattern_out = os.path.join(output_path, output_pattern)
        
        #get lig files
        ligand_files = sorted(glob.glob(full_pattern_lig))
        
        #create the output dir
        if not os.path.exists(output_path):
            os.makedirs(output_path)
            print(f"Directory {output_path} created.")
        
        #error if input wrong
        if not os.path.exists(full_path_prot):
            print(f"Error: Protein file {full_path_prot} not found!")
            sys.exit(1)
        
        if not ligand_files:
            print(f"Error: No ligand files found matching pattern {full_pattern_lig}")
            sys.exit(1)
        
        print(f"Found {len(ligand_files)} ligand files.")
        #combine_pdbs(protein_file, ligand_files, output_pattern)
        combine_with_biopython(full_path_prot, ligand_files, full_pattern_out)
        #combine_structures_direct(full_path_prot, ligand_files, full_pattern_out)
        print("All files combined successfully!")

In [17]:
# protein.pdb "ligand_*.pdb" "complex_*.pdb"
prot_file = "bcsin_233_HPPD_BCS-CV49374_-_prepared.pdb"
ligand_pattern = "BCS-CV49374_*.pdb"
output_pattern = "BCS-CV49374_*.pdb"


# shouldn't change that much 
protein_path = r"../../data/raw/docking/Schrodinger_prep-glide_dock/dock_bcsin_233_BCS-CV49374/"
ligand_path = r"../../data/raw/docking/Schrodinger_prep-glide_dock/dock_bcsin_233_BCS-CV49374/"
output_path = r"../../data/interim/docking/results_SCH_SCH/BCS-CV49374"

# Action
create_combined_res(prot_file, protein_path, ligand_pattern, ligand_path, output_pattern, output_path)

Found 7 ligand files.
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_1.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_2.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_3.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_4.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_5.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_6.pdb
Created combined file: ../../data/interim/docking/results_SCH_SCH/BCS-CV49374/BCS-CV49374_7.pdb
All files combined successfully!


In [27]:
from Bio.PDB import PDBParser, PDBIO, Structure, Model, Chain, Atom

def adjust_ids(structure, offset):
    """Adjust the residue and atom IDs in the structure by a given offset."""
    for model in structure:
        for chain in model:
            for residue in chain:
                # Adjust residue ID
                residue.id = (residue.id[0], residue.id[1] + offset, residue.id[2])
                for atom in residue:
                    # Adjust atom ID
                    atom.id = (atom.id[0], atom.id[1] + offset, atom.id[2])

def merge_pdb_files(file1, file2, output_file):
    # Create a PDBParser
    parser = PDBParser(QUIET=True)
    
    # Load the first PDB file
    structure1 = parser.get_structure('structure1', file1)
    
    # Load the second PDB file
    structure2 = parser.get_structure('structure2', file2)
    
    # Get the number of residues in the first structure to determine the offset
    offset = sum(len(residue) for model in structure1 for chain in model for residue in chain)
    
    # Adjust IDs in the second structure
    adjust_ids(structure2, offset)
    
    # Merge the structures
    for model in structure2:
        for chain in model:
            # Add each chain from structure2 to structure1
            structure1[0].add(chain.copy())  # Use copy to avoid modifying the original structure
    
    # Create a PDBIO to write the merged structure to a file
    io = PDBIO()
    io.set_structure(structure1)
    io.save(output_file)


In [28]:


file1= r"../../data/raw/docking/Schrodinger_prep-glide_dock/dock_bcsin_233_BCS-CV49374/bcsin_233_HPPD_BCS-CV49374_-_prepared.pdb"
file2= r"../../data/raw/docking/Schrodinger_prep-glide_dock/dock_bcsin_233_BCS-CV49374/BCS-CV49374_1.pdb"
output_file = r"../../data/interim/docking/results_SCH_SCH/test.pdb"

merge_pdb_files(file1, file2, output_file)

TypeError: can only concatenate str (not "int") to str